In [5]:
# Definition on auc roc, what am I actually optimising?

In [6]:
import numpy as np
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import kagglehub
import os

In [7]:
url = "https://www.kaggle.com/competitions/playground-series-s6e3/data"
test = "test.csv"
train = "train.csv"

In [8]:
# 1. Download the competition dataset
# This will download the files and return the local path where they are cached
dataset_path = kagglehub.competition_download("playground-series-s6e3")
print(f"Data downloaded to: {dataset_path}")

# 2. Load the train and test CSV files into pandas DataFrames
train_df = pd.read_csv(os.path.join(dataset_path, "train.csv"))
test_df = pd.read_csv(os.path.join(dataset_path, "test.csv"))

# Show the first few rows to verify
train_df

# Check the balance of the target variable. Churn 1 is 22.5%
#

Data downloaded to: C:\Users\zak\.cache\kagglehub\competitions\playground-series-s6e3


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594189,594189,Male,0,No,No,57,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Two year,No,Bank transfer (automatic),97.55,5460.70,No
594190,594190,Female,0,No,No,72,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,Two year,No,Bank transfer (automatic),91.95,6782.15,No
594191,594191,Female,0,Yes,No,72,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Credit card (automatic),24.40,1871.90,No
594192,594192,Female,0,No,No,32,Yes,Yes,Fiber optic,No,...,No,No,No,Yes,Month-to-month,Yes,Electronic check,86.00,2847.20,No


In [10]:
import plotly.express as px

In [11]:
fig = px.histogram(train_df, x='tenure', color='Churn', marginal='box')
fig.show()

In [12]:
X = train_df.drop('Churn', axis=1)
y = train_df['Churn'].values

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

In [14]:
# 1. Define specific transformers for each data type
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

# 2. Use ColumnTransformer to apply them to the right columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, make_column_selector(dtype_include=np.number)),
        ('cat', categorical_transformer, make_column_selector(dtype_include=object))
    ])

log_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression())
])

log_pipe.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [15]:
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold

In [16]:
from sklearn.model_selection import cross_validate

In [17]:
import pandas as pd

# Re-run with explicit flag
results = cross_validate(log_pipe, X, y, cv=5,
                         scoring='roc_auc',
                         return_train_score=True)

# Convert to DataFrame for a much cleaner view
results_df = pd.DataFrame(results)

# Now you can see everything clearly
print(results_df)

# To see just the keys if you're curious:
print("\nAvailable keys:", list(results.keys()))

   fit_time  score_time  test_score  train_score
0  2.754573    0.362246    0.909091     0.907680
1  2.485344    0.369352    0.907659     0.908038
2  2.340426    0.366054    0.907458     0.908078
3  2.404449    0.479731    0.908515     0.907866
4  3.428154    0.600915    0.907044     0.908235

Available keys: ['fit_time', 'score_time', 'test_score', 'train_score']


In [18]:
log_pipe.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [19]:
test_df['Churn'] = log_pipe.predict_proba(test_df)[:, 1]
submission = test_df[['id', 'Churn']]
submission.to_csv('log_pipe.csv', index=False, header=True)

In [20]:
# !kaggle competitions submit -c playground-series-s6e3 -f log_pipe.csv -m "simple logistic regression baseline"

In [21]:
!kaggle competitions submissions -c playground-series-s6e3

fileName               date                        description                          status                     publicScore  privateScore  
---------------------  --------------------------  -----------------------------------  -------------------------  -----------  ------------  
final_sota_blend.csv   2026-03-21 03:00:50.713000  Final SOTA LGBM+XGB blend            SubmissionStatus.COMPLETE  0.91382                    
hgb_pipe_improved.csv  2026-03-21 01:35:39.650000  HistGradientBoostingClassifier       SubmissionStatus.COMPLETE  0.91296                    
hgb_pipe_improved.csv  2026-03-21 01:31:42.733000  HistGradientBoostingClassifier       SubmissionStatus.COMPLETE  0.91205                    
hgb_pipe_improved.csv  2026-03-21 01:29:10.980000  HistGradientBoostingClassifier       SubmissionStatus.COMPLETE  0.91205                    
hgb_pipe_improved.csv  2026-03-21 01:22:59.833000  HistGradientBoostingClassifier       SubmissionStatus.COMPLETE  0.91344                    

After the first model, the public score is about as bad as it gets. Let's diagnose the issue.

- The current best kaggle score is: 0.91758
- Top 100 score is: 0.91702
- 1000 score is: 0.91399

AI suggests that my model is too simple and is probably underfitting. Let's add poly nomial features to the model.

In [22]:
def model_results(pipeline, X, y, cv=5, sample=None):
    """
    Evaluate a pipeline using cross-validation, fit on the full (sampled) data,
    and return the sampled data and fitted model.
    """
    if sample is not None:
        if isinstance(sample, float):
            X = X.sample(frac=sample, random_state=42)
        elif isinstance(sample, int):
            X = X.sample(n=sample, random_state=42)
        y = y[X.index]

    results = cross_validate(
        pipeline, X, y,
        cv=cv,
        scoring='roc_auc',
        return_train_score=True
    )
    
    # Calculate means and standard deviations
    train_mean = results['train_score'].mean()
    train_std = results['train_score'].std()
    test_mean = results['test_score'].mean()
    test_std = results['test_score'].std()
    
    # Print formatted results
    print(f"Train Score: {train_mean:.5f} ± {train_std:.5f}")
    print(f"Test Score:  {test_mean:.5f} ± {test_std:.5f}")
    print(f"Overfit Gap: {train_mean - test_mean:.5f}")

    # Fit once on the full (sampled) dataset to be used for diagnostic plots
    pipeline.fit(X, y)

    return X, y, pipeline

### Experiment 2 (Random Forest)
Let's try a different architecture using Random Forest to see if it captures different patterns in the data.

In [23]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

In [24]:


# 1. Preprocess the target
y = train_df['Churn'].map({'No': 0, 'Yes': 1}).values
X = train_df.drop(['Churn', 'id'], axis=1)

# 2. Define transformers
# RFC handles ordinal features well and it reduces the feature space complexity
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

# 3. Use ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, make_column_selector(dtype_include=np.number)),
        ('cat', categorical_transformer, make_column_selector(dtype_include=object))
    ])

# 4. Create the pipeline with class_weight='balanced'
rfc_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1))
])

# 5. Define a search space for RandomizedSearchCV
param_distributions = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [None, 10, 20, 30],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__max_features': ['sqrt', 'log2']
}

# 6. Execute RandomizedSearchCV
random_search = RandomizedSearchCV(
    rfc_pipe, 
    param_distributions=param_distributions, 
    n_iter=10, 
    cv=3, 
    scoring='roc_auc', 
    verbose=3,
    random_state=42, 
    n_jobs=-1
)

# Use a sample for tuning to speed up the process
X_sample = X.sample(frac=0.1, random_state=42)
y_sample = y[X_sample.index]

random_search.fit(X_sample, y_sample)
print(f"Best parameters: {random_search.best_params_}")
print(f"Best ROC-AUC: {random_search.best_score_:.5f}")

# Update rfc_pipe with the best estimator
rfc_pipe = random_search.best_estimator_

# 7. Evaluate with the full data (or a larger sample)
model_results(rfc_pipe, X, y, cv=5)
# %%
# Predict and create submission
# Ensure the test set also has 'id' dropped for prediction
test_X = test_df.drop('id', axis=1)
test_df['Churn'] = rfc_pipe.predict_proba(test_X)[:, 1]
submission = test_df[['id', 'Churn']]
submission.to_csv('rfc_pipe_improved.csv', index=False, header=True)
# %%

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best parameters: {'classifier__n_estimators': 200, 'classifier__min_samples_split': 2, 'classifier__min_samples_leaf': 4, 'classifier__max_features': 'sqrt', 'classifier__max_depth': 10}
Best ROC-AUC: 0.91060
Train Score: 0.91507 ± 0.00026
Test Score:  0.91197 ± 0.00111
Overfit Gap: 0.00310


In [25]:
# !kaggle competitions submit -c playground-series-s6e3 -f rfc_pipe_improved.csv -m "RandomForestClassifier submission"
!kaggle competitions submissions -c playground-series-s6e3

fileName               date                        description                          status                     publicScore  privateScore  
---------------------  --------------------------  -----------------------------------  -------------------------  -----------  ------------  
final_sota_blend.csv   2026-03-21 03:00:50.713000  Final SOTA LGBM+XGB blend            SubmissionStatus.COMPLETE  0.91382                    
hgb_pipe_improved.csv  2026-03-21 01:35:39.650000  HistGradientBoostingClassifier       SubmissionStatus.COMPLETE  0.91296                    
hgb_pipe_improved.csv  2026-03-21 01:31:42.733000  HistGradientBoostingClassifier       SubmissionStatus.COMPLETE  0.91205                    
hgb_pipe_improved.csv  2026-03-21 01:29:10.980000  HistGradientBoostingClassifier       SubmissionStatus.COMPLETE  0.91205                    
hgb_pipe_improved.csv  2026-03-21 01:22:59.833000  HistGradientBoostingClassifier       SubmissionStatus.COMPLETE  0.91344                    

In [26]:
# ### Experiment 3 (LightGBM AMD GPU)
# The user has a Radeon 6900XT. HistGradientBoostingClassifier is CPU-only, and XGBoost 'device=cuda' is for NVIDIA. 
# We switch to LightGBM, which supports AMD GPUs via OpenCL.
# %%
from lightgbm import LGBMClassifier
from lightgbm import log_evaluation
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# 1. Target and Feature separation
y = train_df['Churn'].map({'No': 0, 'Yes': 1}).values
X = train_df.drop(['Churn', 'id'], axis=1)

# 2. Simple Feature Engineering
def add_features(df):
    df = df.copy()
    # Adding some interaction features
    df['Monthly_Tenure_Ratio'] = df['TotalCharges'] / (df['tenure'] + 1)
    df['Expected_Total_Charges'] = df['MonthlyCharges'] * df['tenure']
    df['Charges_Diff'] = df['TotalCharges'] - df['Expected_Total_Charges']
    return df

X = add_features(X)

# 3. Identify categorical and numeric columns
cat_cols = X.select_dtypes(include=['object', 'str']).columns.tolist()
num_cols = X.select_dtypes(include=np.number).columns.tolist()

# 4. Define transformers
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

# 5. Use ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ])

# 6. Create the pipeline
# device='gpu' uses OpenCL by default, which is compatible with Radeon 6900XT
lgbm_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LGBMClassifier(
        random_state=42,
        device='cpu',
        # gpu_platform_id=0, # These might need adjustment based on user environment
        # gpu_device_id=0,
        importance_type='gain',
        n_jobs=-1
    ))
])

# 7. Define an expanded search space for RandomizedSearchCV
param_distributions = {
    'classifier__n_estimators': [500, 1000, 1500],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__num_leaves': [31, 63, 127],
    'classifier__max_depth': [-1, 5, 10, 15],
    'classifier__min_child_samples': [20, 50, 100],
    'classifier__subsample': [0.6, 0.8, 1.0],
    'classifier__colsample_bytree': [0.6, 0.8, 1.0],
    'classifier__reg_alpha': [0, 0.1, 1],
    'classifier__reg_lambda': [0, 0.1, 1]
}

# 8. Execute RandomizedSearchCV
# Use a 10% sample for tuning
X_sample = X.sample(frac=0.1, random_state=42)
y_sample = y[X_sample.index]

random_search_lgbm = RandomizedSearchCV(
    lgbm_pipe,
    param_distributions=param_distributions,
    n_iter=30,
    cv=3,
    scoring='roc_auc',
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search_lgbm.fit(X_sample, y_sample)
print(f"Best parameters: {random_search_lgbm.best_params_}")
print(f"Best ROC-AUC (tuning sample): {random_search_lgbm.best_score_:.5f}")

# Update lgbm_pipe with the best estimator
lgbm_pipe = random_search_lgbm.best_estimator_

# 9. Evaluate with the full data (or a larger sample)
model_results(lgbm_pipe, X, y, cv=5)

# %%
# Predict and create submission
# Ensure test features match train features
test_X = test_df.drop('id', axis=1)
test_X = add_features(test_X)

# Fit on FULL training data for maximum performance on submission
print("Fitting model on full dataset...")
# log_evaluation(period=100) will print progress every 100 iterations
lgbm_pipe.fit(X, y, classifier__callbacks=[log_evaluation(period=100)])
lgbm_pipe.fit(X, y)

test_df['Churn'] = lgbm_pipe.predict_proba(test_X)[:, 1]
submission_lgbm = test_df[['id', 'Churn']]
submission_lgbm.to_csv('lgbm_pipe_amd_gpu.csv', index=False, header=True)

Fitting 3 folds for each of 30 candidates, totalling 90 fits
[LightGBM] [Info] Number of positive: 13491, number of negative: 45928
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001267 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1392
[LightGBM] [Info] Number of data points in the train set: 59419, number of used features: 22
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.227049 -> initscore=-1.225052
[LightGBM] [Info] Start training from score -1.225052
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Light

C:\Users\zak\Projects\PyCharmProjects\data-science\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names

C:\Users\zak\Projects\PyCharmProjects\data-science\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names



[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009487 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1392
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 22
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225209 -> initscore=-1.235567
[LightGBM] [Info] Start training from score -1.235567
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

C:\Users\zak\Projects\PyCharmProjects\data-science\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names

C:\Users\zak\Projects\PyCharmProjects\data-science\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names



[LightGBM] [Info] Number of positive: 107053, number of negative: 368302
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011037 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1392
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 22
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225206 -> initscore=-1.235579
[LightGBM] [Info] Start training from score -1.235579
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

...

C:\Users\zak\Projects\PyCharmProjects\data-science\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names

C:\Users\zak\Projects\PyCharmProjects\data-science\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names



[LightGBM] [Info] Number of positive: 107054, number of negative: 368302
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008378 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1392
[LightGBM] [Info] Number of data points in the train set: 475356, number of used features: 22
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225208 -> initscore=-1.235570
[LightGBM] [Info] Start training from score -1.235570
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

C:\Users\zak\Projects\PyCharmProjects\data-science\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names

C:\Users\zak\Projects\PyCharmProjects\data-science\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names



Train Score: 0.91653 ± 0.00022
Test Score:  0.91515 ± 0.00101
Overfit Gap: 0.00139
[LightGBM] [Info] Number of positive: 133817, number of negative: 460377
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011509 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1392
[LightGBM] [Info] Number of data points in the train set: 594194, number of used features: 22
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225208 -> initscore=-1.235573
[LightGBM] [Info] Start training from score -1.235573
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

C:\Users\zak\Projects\PyCharmProjects\data-science\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names



In [27]:
# Best parameters (HGB): {'classifier__min_samples_leaf': 100, 'classifier__max_leaf_nodes': 31, 'classifier__max_iter': 500, 'classifier__max_depth': 5, 'classifier__learning_rate': 0.1, 'classifier__l2_regularization': 10}

In [28]:
# !kaggle competitions submit -c playground-series-s6e3 -f lgbm_pipe_amd_gpu.csv -m "LightGBM AMD GPU submission"
!kaggle competitions submissions -c playground-series-s6e3

fileName               date                        description                          status                     publicScore  privateScore  
---------------------  --------------------------  -----------------------------------  -------------------------  -----------  ------------  
final_sota_blend.csv   2026-03-21 03:00:50.713000  Final SOTA LGBM+XGB blend            SubmissionStatus.COMPLETE  0.91382                    
hgb_pipe_improved.csv  2026-03-21 01:35:39.650000  HistGradientBoostingClassifier       SubmissionStatus.COMPLETE  0.91296                    
hgb_pipe_improved.csv  2026-03-21 01:31:42.733000  HistGradientBoostingClassifier       SubmissionStatus.COMPLETE  0.91205                    
hgb_pipe_improved.csv  2026-03-21 01:29:10.980000  HistGradientBoostingClassifier       SubmissionStatus.COMPLETE  0.91205                    
hgb_pipe_improved.csv  2026-03-21 01:22:59.833000  HistGradientBoostingClassifier       SubmissionStatus.COMPLETE  0.91344                    

In [29]:
# Final Model

# State-of-the-art tabular ensemble (LightGBM + XGBoost with OOF blend optimization)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from lightgbm import early_stopping
from xgboost import XGBClassifier


def add_final_features(df):
    df = df.copy()

    # Reuse strong domain interactions and add a few stability-safe transformations
    df['Monthly_Tenure_Ratio'] = df['TotalCharges'] / (df['tenure'] + 1)
    df['Expected_Total_Charges'] = df['MonthlyCharges'] * df['tenure']
    df['Charges_Diff'] = df['TotalCharges'] - df['Expected_Total_Charges']
    df['Monthly_to_Total_Ratio'] = df['MonthlyCharges'] / (df['TotalCharges'] + 1)
    df['Log_TotalCharges'] = np.log1p(np.clip(df['TotalCharges'], a_min=0, a_max=None))
    df['Log_MonthlyCharges'] = np.log1p(np.clip(df['MonthlyCharges'], a_min=0, a_max=None))

    return df


# 1) Prepare data
y_final = train_df['Churn'].map({'No': 0, 'Yes': 1}).values
X_train_final = train_df.drop(['Churn', 'id'], axis=1)
X_test_final = test_df.drop('id', axis=1)

X_train_final = add_final_features(X_train_final)
X_test_final = add_final_features(X_test_final)

# One-hot encode categoricals on concatenated train+test to guarantee column alignment
X_all = pd.concat([X_train_final, X_test_final], axis=0)
X_all = pd.get_dummies(X_all, drop_first=False)

X_train_final = X_all.iloc[:len(X_train_final), :].astype(np.float32)
X_test_final = X_all.iloc[len(X_train_final):, :].astype(np.float32)


# 2) Cross-validated OOF training for robust blend selection
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lgbm = np.zeros(len(X_train_final), dtype=np.float64)
oof_xgb = np.zeros(len(X_train_final), dtype=np.float64)

pred_lgbm_test = np.zeros(len(X_test_final), dtype=np.float64)
pred_xgb_test = np.zeros(len(X_test_final), dtype=np.float64)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train_final, y_final), start=1):
    X_tr, X_va = X_train_final.iloc[tr_idx], X_train_final.iloc[va_idx]
    y_tr, y_va = y_final[tr_idx], y_final[va_idx]

    pos_weight = (len(y_tr) - y_tr.sum()) / y_tr.sum()

    lgbm_final = LGBMClassifier(
        objective='binary',
        random_state=42 + fold,
        n_estimators=4000,
        learning_rate=0.01,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=40,
        subsample=0.85,
        colsample_bytree=0.80,
        reg_alpha=0.5,
        reg_lambda=1.5,
        n_jobs=-1
    )

    xgb_final = XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        random_state=42 + fold,
        n_estimators=5000,
        learning_rate=0.01,
        max_depth=6,
        min_child_weight=4,
        subsample=0.85,
        colsample_bytree=0.75,
        gamma=0.1,
        reg_alpha=0.5,
        reg_lambda=2.0,
        scale_pos_weight=pos_weight,
        early_stopping_rounds=200,
        tree_method='hist',
        n_jobs=-1
    )

    lgbm_final.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric='auc',
        callbacks=[early_stopping(stopping_rounds=200, verbose=False)]
    )

    xgb_final.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False
    )

    oof_lgbm[va_idx] = lgbm_final.predict_proba(X_va)[:, 1]
    oof_xgb[va_idx] = xgb_final.predict_proba(X_va)[:, 1]

    pred_lgbm_test += lgbm_final.predict_proba(X_test_final)[:, 1] / skf.n_splits
    pred_xgb_test += xgb_final.predict_proba(X_test_final)[:, 1] / skf.n_splits

    fold_lgbm_auc = roc_auc_score(y_va, oof_lgbm[va_idx])
    fold_xgb_auc = roc_auc_score(y_va, oof_xgb[va_idx])
    print(f"Fold {fold} | LGBM AUC: {fold_lgbm_auc:.5f} | XGB AUC: {fold_xgb_auc:.5f}")


# 3) Blend optimization on OOF predictions
lgbm_auc = roc_auc_score(y_final, oof_lgbm)
xgb_auc = roc_auc_score(y_final, oof_xgb)
print(f"\nOOF LGBM ROC-AUC: {lgbm_auc:.5f}")
print(f"OOF XGB ROC-AUC:  {xgb_auc:.5f}")

weights = np.linspace(0.0, 1.0, 101)
best_weight = 0.5
best_auc = 0.0

for w in weights:
    blend_oof = w * oof_lgbm + (1 - w) * oof_xgb
    score = roc_auc_score(y_final, blend_oof)
    if score > best_auc:
        best_auc = score
        best_weight = w

print(f"Best blend weight (LGBM): {best_weight:.2f}")
print(f"Best OOF blend ROC-AUC:   {best_auc:.5f}")


# 4) Build final submission
final_test_pred = best_weight * pred_lgbm_test + (1 - best_weight) * pred_xgb_test

submission_final = test_df[['id']].copy()
submission_final['Churn'] = final_test_pred
submission_final.to_csv('final_sota_blend.csv', index=False)

print("Saved final submission file: final_sota_blend.csv")

# !kaggle competitions submit -c playground-series-s6e3 -f final_sota_blend.csv -m "Final SOTA LGBM+XGB blend"


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008232 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2197
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 51
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225209 -> initscore=-1.235567
[LightGBM] [Info] Start training from score -1.235567
Fold 1 | LGBM AUC: 0.91588 | XGB AUC: 0.91599
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testin

In [ ]:
!kaggle competitions submit -c playground-series-s6e3 -f final_sota_blend.csv -m "Final SOTA LGBM+XGB blend"

In [ ]:
# !kaggle competitions submit -c playground-series-s6e3 -f lgbm_pipe_amd_gpu.csv -m "LightGBM AMD GPU submission"
!kaggle competitions submissions -c playground-series-s6e3

In [ ]:
# assume train_df/test_df already loaded
print(train_df.shape, test_df.shape)
print("Train dup rows:", train_df.duplicated().sum())
print("Test dup rows:", test_df.duplicated().sum())

na_train = train_df.isna().mean().sort_values(ascending=False)
na_test = test_df.isna().mean().sort_values(ascending=False)
print("Top NA train:\n", na_train.head(10))
print("Top NA test:\n", na_test.head(10))

const_cols = [c for c in train_df.columns if train_df[c].nunique(dropna=False) <= 1]
print("Constant columns:", const_cols)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

y = train_df['Churn'].map({'No': 0, 'Yes': 1}).values
X = train_df.drop(columns=['Churn', 'id'], errors='ignore')

y_shuf = np.random.RandomState(42).permutation(y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# reuse your existing pipeline/model here
scores = cross_val_score(log_pipe, X, y_shuf, cv=cv, scoring='roc_auc', n_jobs=-1)
print("Shuffled-label AUC:", scores.mean(), "+/-", scores.std())

In [ ]:
from sklearn.model_selection import cross_val_score

X_train = train_df.drop(columns=['Churn'], errors='ignore').copy()
X_test = test_df.copy()
X_both = pd.concat([X_train, X_test], axis=0, ignore_index=True)
y_adv = np.r_[np.zeros(len(X_train)), np.ones(len(X_test))]

adv_scores = cross_val_score(log_pipe, X_both, y_adv, cv=5, scoring='roc_auc', n_jobs=-1)
print("Adversarial AUC:", adv_scores.mean())

In [ ]:
X_train = train_df.drop(columns=['Churn', 'id'], errors='ignore').copy()
X_test = test_df.drop(columns=['id'], errors='ignore').copy()

X_both = pd.concat([X_train, X_test], axis=0, ignore_index=True)
y_adv = np.r_[np.zeros(len(X_train)), np.ones(len(X_test))]

adv_scores = cross_val_score(log_pipe, X_both, y_adv, cv=5, scoring='roc_auc', n_jobs=-1)
print("Adversarial AUC:", adv_scores.mean(), "+/-", adv_scores.std())

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score


def adversarial_validation_report(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    target_col: str = "Churn",
    id_col: str = "id",
    cv: int = 5,
    random_state: int = 42,
    top_n: int = 20,
):
    """
    Returns:
      - auc_mean: mean CV ROC-AUC for train-vs-test classifier
      - auc_std:  std CV ROC-AUC
      - drift_features: DataFrame of top drifting features
    """

    # 1) Build adversarial dataset (IMPORTANT: drop id and target)
    X_train = train_df.drop(columns=[target_col, id_col], errors="ignore").copy()
    X_test = test_df.drop(columns=[id_col], errors="ignore").copy()

    X_both = pd.concat([X_train, X_test], axis=0, ignore_index=True)
    y_adv = np.r_[np.zeros(len(X_train), dtype=np.int8), np.ones(len(X_test), dtype=np.int8)]

    # 2) Preprocessing (keeps 1 column per original feature)
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, make_column_selector(dtype_include=np.number)),
        ("cat", categorical_transformer, make_column_selector(dtype_exclude=np.number)),
    ])

    adv_model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=600,
            min_samples_leaf=5,
            class_weight="balanced_subsample",
            random_state=random_state,
            n_jobs=-1,
        )),
    ])

    # 3) CV AUC
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)
    scores = cross_val_score(adv_model, X_both, y_adv, cv=skf, scoring="roc_auc", n_jobs=-1)

    # 4) Fit once to get drift feature importances
    adv_model.fit(X_both, y_adv)
    feature_names = adv_model.named_steps["preprocessor"].get_feature_names_out()
    importances = adv_model.named_steps["classifier"].feature_importances_

    drift_features = (
        pd.DataFrame({"feature": feature_names, "importance": importances})
        .assign(feature=lambda d: d["feature"].str.replace(r"^(num|cat)__", "", regex=True))
        .sort_values("importance", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    return scores.mean(), scores.std(), drift_features

In [ ]:
auc_mean, auc_std, drift_df = adversarial_validation_report(
    train_df=train_df,
    test_df=test_df,
    target_col="Churn",
    id_col="id",
    cv=5,
    random_state=42,
    top_n=20,
)

print(f"Adversarial AUC: {auc_mean:.5f} +/- {auc_std:.5f}")
print("Top drift features:")
print(drift_df)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report, f1_score

# 1) Build OOF blended probabilities
oof_blend = best_weight * oof_lgbm + (1 - best_weight) * oof_xgb
y_true = y_final.astype(int)

# 2) Pick a threshold (don't assume 0.5)
# Here: choose threshold that maximizes F1 on OOF
ths = np.linspace(0.01, 0.99, 199)
f1s = [f1_score(y_true, (oof_blend >= t).astype(int)) for t in ths]
best_t = float(ths[int(np.argmax(f1s))])

print(f"Chosen threshold (OOF F1-optimal): {best_t:.3f}")

# 3) Build analysis dataframe on original train features
err_df = train_df.drop(columns=['Churn']).copy()
err_df['y_true'] = y_true
err_df['p_churn'] = oof_blend
err_df['y_pred'] = (err_df['p_churn'] >= best_t).astype(int)
err_df['error'] = (err_df['y_true'] != err_df['y_pred'])

err_df['error_type'] = np.select(
    [
        (err_df['y_true'] == 1) & (err_df['y_pred'] == 0),  # false negative
        (err_df['y_true'] == 0) & (err_df['y_pred'] == 1),  # false positive
    ],
    ['FN', 'FP'],
    default='OK'
)

# confidence relative to threshold: bigger => model was more sure
err_df['confidence'] = np.abs(err_df['p_churn'] - best_t)
err_df['confident_wrong'] = err_df['error'] & (err_df['confidence'] >= 0.20)
err_df['borderline'] = err_df['error'] & (err_df['confidence'] <= 0.05)

# 4) Global summary
cm = confusion_matrix(err_df['y_true'], err_df['y_pred'])
print("Confusion matrix [[TN, FP], [FN, TP]]:\n", cm)
print(classification_report(err_df['y_true'], err_df['y_pred'], digits=4))
print("Total error rate:", err_df['error'].mean().round(4))
print(err_df['error_type'].value_counts(dropna=False))
print("Confident wrong count:", int(err_df['confident_wrong'].sum()))
print("Borderline wrong count:", int(err_df['borderline'].sum()))

# 5) Most important rows to inspect manually (high-confidence mistakes)
cols_to_show = [c for c in ['id', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Contract', 'InternetService', 'PaymentMethod'] if c in err_df.columns]
print("\nTop confident mistakes:")
display(
    err_df.loc[err_df['error']]
          .sort_values('confidence', ascending=False)
          .head(25)[cols_to_show + ['y_true', 'p_churn', 'y_pred', 'error_type', 'confidence']]
)

In [ ]:
def segment_error_table(df, col, min_count=150, q=10):
    d = df.copy()
    group_col = col

    if pd.api.types.is_numeric_dtype(d[col]):
        # quantile bins for numeric columns
        n_unique = d[col].nunique(dropna=True)
        n_bins = int(min(q, max(2, n_unique)))
        d[col + '_bin'] = pd.qcut(d[col], q=n_bins, duplicates='drop')
        group_col = col + '_bin'

    out = (
        d.groupby(group_col)
         .agg(
             n=('error', 'size'),
             error_rate=('error', 'mean'),
             fn_rate=('error_type', lambda s: (s == 'FN').mean()),
             fp_rate=('error_type', lambda s: (s == 'FP').mean()),
             avg_p=('p_churn', 'mean')
         )
         .query('n >= @min_count')
         .sort_values('error_rate', ascending=False)
    )
    return out

for c in ['Contract', 'InternetService', 'PaymentMethod', 'tenure', 'MonthlyCharges', 'TotalCharges']:
    if c in err_df.columns:
        print(f"\n=== {c} ===")
        display(segment_error_table(err_df, c).head(10))

In [ ]:
import numpy as np
import pandas as pd


def misclassification_feature_report(
    err_df: pd.DataFrame,
    feature_cols=None,
    top_n: int = 20,
    min_n: int = 80,
    max_levels: int = 30,
):
    """
    Compares feature distributions for:
      1) FN vs TP  (missed churners vs correctly detected churners)
      2) FP vs TN  (false alarms vs correctly detected non-churners)

    Returns a dict of ranked DataFrames.
    """

    # Columns created during evaluation that should not be analyzed as predictors
    exclude_cols = {
        'y_true', 'y_pred', 'p_churn', 'error', 'error_type',
        'confidence', 'confident_wrong', 'borderline'
    }

    d = err_df.copy()

    if feature_cols is None:
        feature_cols = [c for c in d.columns if c not in exclude_cols]

    num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(d[c])]
    cat_cols = [c for c in feature_cols if c not in num_cols]

    # Masks
    fn = (d['y_true'] == 1) & (d['y_pred'] == 0)
    tp = (d['y_true'] == 1) & (d['y_pred'] == 1)
    fp = (d['y_true'] == 0) & (d['y_pred'] == 1)
    tn = (d['y_true'] == 0) & (d['y_pred'] == 0)

    def _numeric_compare(mask_a, mask_b, cols, label_a, label_b):
        rows = []
        for c in cols:
            a = d.loc[mask_a, c].dropna().astype(float)
            b = d.loc[mask_b, c].dropna().astype(float)

            if len(a) < min_n or len(b) < min_n:
                continue

            mean_a, mean_b = a.mean(), b.mean()
            med_a, med_b = a.median(), b.median()
            std_a, std_b = a.std(ddof=1), b.std(ddof=1)

            pooled = np.sqrt((std_a**2 + std_b**2) / 2.0)
            smd = (mean_a - mean_b) / (pooled + 1e-12)  # standardized mean diff (effect size)

            q1_a, q1_b = a.quantile(0.25), b.quantile(0.25)
            q3_a, q3_b = a.quantile(0.75), b.quantile(0.75)

            rows.append({
                'feature': c,
                'n_' + label_a: len(a),
                'n_' + label_b: len(b),
                'mean_' + label_a: mean_a,
                'mean_' + label_b: mean_b,
                'median_' + label_a: med_a,
                'median_' + label_b: med_b,
                'mean_diff': mean_a - mean_b,
                'median_diff': med_a - med_b,
                'smd': smd,
                'abs_smd': abs(smd),
                'q25_diff': q1_a - q1_b,
                'q75_diff': q3_a - q3_b,
            })

        out = pd.DataFrame(rows)
        if len(out) == 0:
            return out
        return out.sort_values('abs_smd', ascending=False).head(top_n).reset_index(drop=True)

    def _categorical_compare(mask_a, mask_b, cols, label_a, label_b):
        rows = []
        for c in cols:
            a = d.loc[mask_a, c].fillna('__MISSING__').astype(str)
            b = d.loc[mask_b, c].fillna('__MISSING__').astype(str)

            if len(a) < min_n or len(b) < min_n:
                continue

            # Optional level cap to avoid extremely high-cardinality noise
            top_levels = (
                pd.concat([a, b], axis=0)
                .value_counts()
                .head(max_levels)
                .index
            )
            a2 = a.where(a.isin(top_levels), '__OTHER__')
            b2 = b.where(b.isin(top_levels), '__OTHER__')

            pa = a2.value_counts(normalize=True)
            pb = b2.value_counts(normalize=True)
            idx = pa.index.union(pb.index)
            pa = pa.reindex(idx, fill_value=0.0)
            pb = pb.reindex(idx, fill_value=0.0)

            # Total variation distance in [0,1], higher = stronger distribution shift
            tvd = 0.5 * np.abs(pa - pb).sum()

            # Top shifted levels
            lvl_shift = (pa - pb).abs().sort_values(ascending=False).head(3)
            top_shift_str = '; '.join([f"{lvl}: {float(pa[lvl]-pb[lvl]):+.3f}" for lvl in lvl_shift.index])

            rows.append({
                'feature': c,
                'n_' + label_a: len(a),
                'n_' + label_b: len(b),
                'tvd': float(tvd),
                'top_shift_levels (a-b)': top_shift_str,
            })

        out = pd.DataFrame(rows)
        if len(out) == 0:
            return out
        return out.sort_values('tvd', ascending=False).head(top_n).reset_index(drop=True)

    report = {
        'counts': {
            'FN': int(fn.sum()), 'TP': int(tp.sum()),
            'FP': int(fp.sum()), 'TN': int(tn.sum())
        },
        'fn_vs_tp_numeric': _numeric_compare(fn, tp, num_cols, 'fn', 'tp'),
        'fn_vs_tp_categorical': _categorical_compare(fn, tp, cat_cols, 'fn', 'tp'),
        'fp_vs_tn_numeric': _numeric_compare(fp, tn, num_cols, 'fp', 'tn'),
        'fp_vs_tn_categorical': _categorical_compare(fp, tn, cat_cols, 'fp', 'tn'),
    }

    return report

In [ ]:
report = misclassification_feature_report(
    err_df,
    top_n=20,
    min_n=80,      # lower if you want more features to pass
    max_levels=30  # cap high-cardinality categoricals
)

print("Group counts:", report['counts'])

print("\n=== FN vs TP (numeric) ===")
display(report['fn_vs_tp_numeric'])

print("\n=== FN vs TP (categorical) ===")
display(report['fn_vs_tp_categorical'])

print("\n=== FP vs TN (numeric) ===")
display(report['fp_vs_tn_numeric'])

print("\n=== FP vs TN (categorical) ===")
display(report['fp_vs_tn_categorical'])

In [ ]:
import numpy as np
import pandas as pd


def _safe_head(df, n):
    if df is None or len(df) == 0:
        return pd.DataFrame()
    return df.head(n).copy()


def threshold_sweep_report(
    err_df: pd.DataFrame,
    y_col: str = "y_true",
    p_col: str = "p_churn",
    thresholds=None,
    fn_cost: float = 3.0,
    fp_cost: float = 1.0,
):
    """
    Evaluate threshold trade-offs on OOF probabilities.
    Returns per-threshold metrics DataFrame.
    """
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 197)

    y = err_df[y_col].astype(int).values
    p = err_df[p_col].astype(float).values

    rows = []
    for t in thresholds:
        pred = (p >= t).astype(int)

        tp = int(((pred == 1) & (y == 1)).sum())
        fp = int(((pred == 1) & (y == 0)).sum())
        fn = int(((pred == 0) & (y == 1)).sum())
        tn = int(((pred == 0) & (y == 0)).sum())

        precision = tp / (tp + fp + 1e-12)
        recall = tp / (tp + fn + 1e-12)
        f1 = 2 * precision * recall / (precision + recall + 1e-12)
        error_rate = (fp + fn) / len(y)
        weighted_cost = fn_cost * fn + fp_cost * fp

        rows.append({
            "threshold": float(t),
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "tn": tn,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "error_rate": error_rate,
            "weighted_cost": weighted_cost,
        })

    return pd.DataFrame(rows)


def generate_feature_ideas_from_report(
    report: dict,
    top_n: int = 8,
    smd_cutoff: float = 0.12,
    tvd_cutoff: float = 0.05,
):
    """
    Turn FN/FP driver tables into actionable feature-engineering ideas.
    """
    ideas = []

    # FN vs TP numeric -> help catch missed churners
    fn_num = _safe_head(report.get("fn_vs_tp_numeric", pd.DataFrame()), top_n)
    if len(fn_num):
        for _, r in fn_num.iterrows():
            if float(r.get("abs_smd", 0.0)) < smd_cutoff:
                continue
            f = r["feature"]
            direction = "higher" if float(r.get("smd", 0.0)) > 0 else "lower"
            ideas.append({
                "priority": abs(float(r.get("smd", 0.0))),
                "focus": "Reduce FN",
                "feature": f,
                "signal": f"{f} is {direction} in FN vs TP",
                "suggested_action": "Add non-linear transform/binning + interactions to recover missed churners.",
                "candidate_features": f"{f}_bin, log1p({f}) (if skewed), {f}_x_tenure, {f}_x_Contract",
            })

    # FN vs TP categorical -> help catch missed churners in specific levels
    fn_cat = _safe_head(report.get("fn_vs_tp_categorical", pd.DataFrame()), top_n)
    if len(fn_cat):
        for _, r in fn_cat.iterrows():
            if float(r.get("tvd", 0.0)) < tvd_cutoff:
                continue
            f = r["feature"]
            lvl = str(r.get("top_shift_levels (a-b)", ""))
            ideas.append({
                "priority": float(r.get("tvd", 0.0)),
                "focus": "Reduce FN",
                "feature": f,
                "signal": f"Shifted levels in FN: {lvl}",
                "suggested_action": "Group high-risk levels and add interactions with price/tenure features.",
                "candidate_features": f"is_{f}_risk_level, {f}_x_MonthlyCharges, {f}_x_tenure",
            })

    # FP vs TN numeric -> reduce false alarms
    fp_num = _safe_head(report.get("fp_vs_tn_numeric", pd.DataFrame()), top_n)
    if len(fp_num):
        for _, r in fp_num.iterrows():
            if float(r.get("abs_smd", 0.0)) < smd_cutoff:
                continue
            f = r["feature"]
            direction = "higher" if float(r.get("smd", 0.0)) > 0 else "lower"
            ideas.append({
                "priority": abs(float(r.get("smd", 0.0))),
                "focus": "Reduce FP",
                "feature": f,
                "signal": f"{f} is {direction} in FP vs TN",
                "suggested_action": "Add stabilizing interactions / piecewise bins to avoid over-triggering.",
                "candidate_features": f"{f}_bin, {f}_x_Contract, {f}_x_InternetService",
            })

    # FP vs TN categorical -> reduce false alarms in specific categories
    fp_cat = _safe_head(report.get("fp_vs_tn_categorical", pd.DataFrame()), top_n)
    if len(fp_cat):
        for _, r in fp_cat.iterrows():
            if float(r.get("tvd", 0.0)) < tvd_cutoff:
                continue
            f = r["feature"]
            lvl = str(r.get("top_shift_levels (a-b)", ""))
            ideas.append({
                "priority": float(r.get("tvd", 0.0)),
                "focus": "Reduce FP",
                "feature": f,
                "signal": f"Shifted levels in FP: {lvl}",
                "suggested_action": "Collapse noisy levels and add cross-features to improve specificity.",
                "candidate_features": f"{f}_grouped, {f}_x_PaymentMethod, {f}_x_tenure",
            })

    ideas_df = pd.DataFrame(ideas)
    if len(ideas_df):
        ideas_df = ideas_df.sort_values("priority", ascending=False).reset_index(drop=True)

    return ideas_df


def auto_error_improvement_plan(
    report: dict,
    err_df: pd.DataFrame,
    base_threshold: float | None = None,
    top_n_ideas: int = 8,
    smd_cutoff: float = 0.12,
    tvd_cutoff: float = 0.05,
    fn_cost: float = 3.0,
    fp_cost: float = 1.0,
):
    ideas_df = generate_feature_ideas_from_report(
        report=report,
        top_n=top_n_ideas,
        smd_cutoff=smd_cutoff,
        tvd_cutoff=tvd_cutoff,
    )

    sweep_df = threshold_sweep_report(
        err_df=err_df,
        fn_cost=fn_cost,
        fp_cost=fp_cost,
    )

    best_f1 = sweep_df.loc[sweep_df["f1"].idxmax()].copy()
    best_cost = sweep_df.loc[sweep_df["weighted_cost"].idxmin()].copy()

    rows = [
        {
            "strategy": "Best F1",
            "threshold": float(best_f1["threshold"]),
            "precision": float(best_f1["precision"]),
            "recall": float(best_f1["recall"]),
            "f1": float(best_f1["f1"]),
            "weighted_cost": float(best_f1["weighted_cost"]),
        },
        {
            "strategy": f"Min Cost (FN:{fn_cost}, FP:{fp_cost})",
            "threshold": float(best_cost["threshold"]),
            "precision": float(best_cost["precision"]),
            "recall": float(best_cost["recall"]),
            "f1": float(best_cost["f1"]),
            "weighted_cost": float(best_cost["weighted_cost"]),
        },
    ]

    if base_threshold is not None:
        base_row = sweep_df.iloc[(sweep_df["threshold"] - base_threshold).abs().argmin()].copy()
        rows.insert(0, {
            "strategy": "Current/Base",
            "threshold": float(base_row["threshold"]),
            "precision": float(base_row["precision"]),
            "recall": float(base_row["recall"]),
            "f1": float(base_row["f1"]),
            "weighted_cost": float(base_row["weighted_cost"]),
        })

    threshold_summary = pd.DataFrame(rows)
    return ideas_df, sweep_df, threshold_summary

In [ ]:
ideas_df, sweep_df, threshold_summary = auto_error_improvement_plan(
    report=report,
    err_df=err_df,
    base_threshold=best_t,   # from your earlier OOF threshold search
    top_n_ideas=8,
    smd_cutoff=0.12,
    tvd_cutoff=0.05,
    fn_cost=3.0,             # increase if missing churners is worse
    fp_cost=1.0,
)

print("=== Suggested feature ideas (priority-ranked) ===")
display(ideas_df.head(20))

print("=== Threshold strategy comparison ===")
display(threshold_summary)

print("=== Top thresholds by weighted cost ===")
display(sweep_df.sort_values("weighted_cost").head(10))

In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from lightgbm import LGBMClassifier, early_stopping
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score


def _unique_keep_order(seq):
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def _extract_driver_features(report, train_raw, top_num=5, top_cat=4):
    num_cols = train_raw.select_dtypes(include=np.number).columns.tolist()
    cat_cols = train_raw.select_dtypes(exclude=np.number).columns.tolist()

    fn_num = report.get('fn_vs_tp_numeric', pd.DataFrame())
    fp_num = report.get('fp_vs_tn_numeric', pd.DataFrame())
    fn_cat = report.get('fn_vs_tp_categorical', pd.DataFrame())
    fp_cat = report.get('fp_vs_tn_categorical', pd.DataFrame())

    num_feats = []
    cat_feats = []

    for df in [fn_num, fp_num]:
        if len(df) and 'feature' in df.columns:
            num_feats.extend(df['feature'].astype(str).tolist())

    for df in [fn_cat, fp_cat]:
        if len(df) and 'feature' in df.columns:
            cat_feats.extend(df['feature'].astype(str).tolist())

    # Keep only valid columns and remove obvious leakage-ish field
    num_feats = [f for f in _unique_keep_order(num_feats) if f in num_cols and f != 'id']
    cat_feats = [f for f in _unique_keep_order(cat_feats) if f in cat_cols and f != 'id']

    # Fallbacks if report tables are sparse
    default_num = [c for c in ['tenure', 'MonthlyCharges', 'TotalCharges'] if c in num_cols]
    default_cat = [c for c in ['Contract', 'InternetService', 'PaymentMethod'] if c in cat_cols]

    num_feats = _unique_keep_order(num_feats + default_num)[:top_num]
    cat_feats = _unique_keep_order(cat_feats + default_cat)[:top_cat]

    return num_feats, cat_feats


def _build_candidate_specs(num_feats, cat_feats):
    specs = []

    # Numeric x Numeric
    for a, b in combinations(num_feats, 2):
        specs.append({'name': f'{a}__x__{b}', 'kind': 'num_mul', 'a': a, 'b': b})
        specs.append({'name': f'{a}__div__{b}', 'kind': 'num_div', 'a': a, 'b': b})

    # Categorical x Categorical
    for a, b in combinations(cat_feats, 2):
        specs.append({'name': f'{a}__X__{b}', 'kind': 'cat_cat', 'a': a, 'b': b})

    # Categorical x Numeric-bin
    for c in cat_feats:
        for n in num_feats:
            specs.append({'name': f'{c}__X__{n}_bin5', 'kind': 'cat_num_bin', 'a': c, 'b': n})

    return specs


def _apply_candidate(df, spec):
    d = df.copy()
    a, b = spec['a'], spec['b']

    if a not in d.columns or b not in d.columns:
        return d

    if spec['kind'] == 'num_mul':
        d[spec['name']] = d[a].astype(float) * d[b].astype(float)

    elif spec['kind'] == 'num_div':
        d[spec['name']] = d[a].astype(float) / (np.abs(d[b].astype(float)) + 1e-3)

    elif spec['kind'] == 'cat_cat':
        d[spec['name']] = d[a].astype(str) + '__' + d[b].astype(str)

    elif spec['kind'] == 'cat_num_bin':
        ranks = d[b].rank(method='first')
        bins = pd.qcut(ranks, q=5, duplicates='drop').astype(str)
        d[spec['name']] = d[a].astype(str) + '__' + bins

    return d


def _build_matrix_with_optional_candidate(X_train_raw, X_test_raw, candidate_spec=None):
    # Uses your existing function from the notebook
    X_tr = add_final_features(X_train_raw.copy())
    X_te = add_final_features(X_test_raw.copy())

    if candidate_spec is not None:
        both = pd.concat([X_tr, X_te], axis=0, ignore_index=True)
        both = _apply_candidate(both, candidate_spec)
        n_tr = len(X_tr)
        X_tr = both.iloc[:n_tr].copy()
        X_te = both.iloc[n_tr:].copy()

    X_all = pd.get_dummies(pd.concat([X_tr, X_te], axis=0, ignore_index=True), drop_first=False)
    n_tr = len(X_tr)
    X_tr_mat = X_all.iloc[:n_tr].astype(np.float32)
    X_te_mat = X_all.iloc[n_tr:].astype(np.float32)
    return X_tr_mat, X_te_mat


def _oof_auc_lgbm(X, y, n_splits=3, random_state=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    oof = np.zeros(len(X), dtype=np.float64)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        model = LGBMClassifier(
            objective='binary',
            random_state=random_state + fold,
            n_estimators=1800,
            learning_rate=0.03,
            num_leaves=63,
            min_child_samples=40,
            subsample=0.85,
            colsample_bytree=0.80,
            reg_alpha=0.5,
            reg_lambda=1.5,
            n_jobs=-1,
        )

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric='auc',
            callbacks=[early_stopping(stopping_rounds=120, verbose=False)]
        )

        oof[va_idx] = model.predict_proba(X_va)[:, 1]

    return float(roc_auc_score(y, oof))


def interaction_candidate_screen(
    train_df,
    test_df,
    y,
    report,
    max_candidates=20,
    top_num=5,
    top_cat=4,
    n_splits_screen=3,
    sample_frac=0.40,
    random_state=42,
):
    """
    Returns:
      base_auc, results_df
    results_df columns:
      candidate, kind, a, b, auc, lift
    """

    y = np.asarray(y).astype(int)

    X_train_raw = train_df.drop(columns=['Churn', 'id'], errors='ignore').copy()
    X_test_raw = test_df.drop(columns=['id'], errors='ignore').copy()

    num_feats, cat_feats = _extract_driver_features(report, X_train_raw, top_num=top_num, top_cat=top_cat)
    specs = _build_candidate_specs(num_feats, cat_feats)
    specs = specs[:max_candidates]

    # Build baseline matrix once
    X_base, _ = _build_matrix_with_optional_candidate(X_train_raw, X_test_raw, candidate_spec=None)

    # Optional stratified sample for faster screening
    if sample_frac < 1.0:
        sss = StratifiedShuffleSplit(n_splits=1, train_size=sample_frac, random_state=random_state)
        sample_idx, _ = next(sss.split(np.zeros(len(y)), y))
    else:
        sample_idx = np.arange(len(y))

    X_base_eval = X_base.iloc[sample_idx].reset_index(drop=True)
    y_eval = y[sample_idx]

    base_auc = _oof_auc_lgbm(X_base_eval, y_eval, n_splits=n_splits_screen, random_state=random_state)
    print(f"Baseline screen AUC: {base_auc:.6f}")
    print(f"Testing {len(specs)} candidates...")

    rows = []
    for i, spec in enumerate(specs, start=1):
        X_cand, _ = _build_matrix_with_optional_candidate(X_train_raw, X_test_raw, candidate_spec=spec)
        X_cand_eval = X_cand.iloc[sample_idx].reset_index(drop=True)

        auc = _oof_auc_lgbm(X_cand_eval, y_eval, n_splits=n_splits_screen, random_state=random_state)
        lift = auc - base_auc

        rows.append({
            'candidate': spec['name'],
            'kind': spec['kind'],
            'a': spec['a'],
            'b': spec['b'],
            'auc': auc,
            'lift': lift,
        })

        print(f"[{i:02d}/{len(specs):02d}] {spec['name']:<40} AUC={auc:.6f} | lift={lift:+.6f}")

    results_df = pd.DataFrame(rows).sort_values('lift', ascending=False).reset_index(drop=True)
    return base_auc, results_df

In [ ]:
base_auc, lift_df = interaction_candidate_screen(
    train_df=train_df,
    test_df=test_df,
    y=y_final,      # from your final model section
    report=report,  # from misclassification_feature_report
    max_candidates=20,
    top_num=5,
    top_cat=4,
    n_splits_screen=3,
    sample_frac=0.40,   # increase to 1.0 for full-data screening
    random_state=42,
)

print("\n=== Top candidates by OOF AUC lift ===")
display(lift_df.head(15))

print("\nCandidates worth promoting to full 5-fold blend test:")
display(lift_df[lift_df['lift'] > 0].head(5))